# Assignment 9


In [84]:
import os
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import json




### Import data

In [ ]:
random.seed(42)



# Path to data
data_dir = "../../MainProject/Assignment9/data/kinect_good_preprocessed"

# Get all CSV files
files = []
for f in os.listdir(data_dir):
    if f.endswith(".csv"):
        files.append(f)


# Split into train/test 
test_files = random.sample(files, 10)
train_files = []
for f in files:
    if f not in test_files:
        train_files.append(f)


# Load data
def load(files):
    dataframes = []

    for f in files:
        path = os.path.join(data_dir, f)   
        df = pd.read_csv(path)           
        dataframes.append(df)             

    combined = pd.concat(dataframes, ignore_index=True)  # combine all

    return combined



train_data = load(train_files)
test_data = load(test_files)



# Split into X (x,y) and y (z)
#  find input columns (x and y)
input_cols = []
for c in train_data.columns:
    if c.endswith("_x") or c.endswith("_y"):
        input_cols.append(c)

# find target columns (z)
target_cols = []
for c in train_data.columns:
    if c.endswith("_z"):
        target_cols.append(c)

X_train = train_data[input_cols].values
y_train = train_data[target_cols].values

X_test = test_data[input_cols].values
y_test = test_data[target_cols].values



# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

### Model

### Saving model

In [ ]:
# Base paths
base_model_dir = "../assignment9_models"
candidates_dir = os.path.join(base_model_dir, "candidates")
champion_dir = os.path.join(base_model_dir, "champion")
metadata_dir = os.path.join(base_model_dir, "metadata")

# Create folders if they don't exist
os.makedirs(candidates_dir, exist_ok=True)
os.makedirs(champion_dir, exist_ok=True)
os.makedirs(metadata_dir, exist_ok=True)


# =========================
# SAVE CANDIDATE MODEL
# =========================
def save_candidate_model(model, model_name):
    path = os.path.join(candidates_dir, f"{model_name}.pt")
    torch.save(model.state_dict(), path)
    print(f"Saved candidate model: {path}")
    return path


# =========================
# LOAD CHAMPION INFO
# =========================
def load_champion_info():
    path = os.path.join(metadata_dir, "champion_info.json")

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None  


# =========================
# SAVE NEW CHAMPION
# =========================
def save_champion_model(model, model_name, mse, mae, hyperparameters):
    model_path = os.path.join(champion_dir, "champion_model.pt")
    info_path = os.path.join(metadata_dir, "champion_info.json")

    # Save model weights
    torch.save(model.state_dict(), model_path)

    # Save metadata
    info = {
        "model_name": model_name,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "mse": float(mse),
        "mae": float(mae),
        "hyperparameters": hyperparameters
    }

    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)

    print("New champion model saved!")


# =========================
# UPDATE CHAMPION LOGIC
# =========================
def update_champion(model, model_name, mse, mae, hyperparameters):
    current = load_champion_info()

    if current is None:
        print("No champion found --> saving first model")
        save_champion_model(model, model_name, mse, mae, hyperparameters)

    elif mae < current["mae"]:
        print(f"New model is better (MSE {mse} < {current['mse']})")
        save_champion_model(model, model_name, mse, mae, hyperparameters)

    else:
        print(f"Model NOT better (MSE {mse} ≥ {current['mse']})")

In [ ]:
torch.manual_seed(42)


hyperparameters = {
    "layers": [128, 32],
    "activation": "ReLU",
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "epochs": 20
}

input_size = X_train.shape[1]
output_size = y_train.shape[1]

model = nn.Sequential(
    nn.Linear(input_size, hyperparameters["layers"][0]),
    nn.ReLU(),
    nn.Linear(hyperparameters["layers"][0], hyperparameters["layers"][1]),
    nn.ReLU(),
    nn.Linear(hyperparameters["layers"][1], output_size)
)

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparameters["learning_rate"])

for epoch in range(hyperparameters["epochs"]):
    predictions = model(X_train)
    loss = loss_fn(predictions, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_pred = model(X_test)
    mse = torch.mean((test_pred - y_test) ** 2)
    mae = torch.mean(torch.abs(test_pred - y_test))





    

print("MSE:", mse.item())
print("MAE:", mae.item())

model_name = "model_v2"

save_candidate_model(model, model_name)

update_champion(
    model,
    model_name,
    mse.item(),
    mae.item(),
    hyperparameters
)

MSE: 0.008560383692383766
MAE: 0.07551944255828857
Saved candidate model: ../assignment9_models/candidates/model_v3.pt
Model NOT better (MSE 0.008560383692383766 ≥ 0.008560383692383766)
